In [ ]:
import os, platform, sys
import numpy as np
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score

# ---------- 1. Informasi lingkungan (untuk placeholder Subbab 2.1) ----------
def info_lingkungan():
    import sklearn, scipy
    print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)
    try:
        import imblearn; print("imbalanced-learn", imblearn.__version__)
    except ImportError:
        print("imbalanced-learn: tidak terpasang (cek pustaka SMOTE yang Anda pakai)")
    print("OS:", platform.platform(), "| CPU:", platform.processor() or platform.machine(), "| core:", os.cpu_count())
    try:
        import psutil; print("RAM: %.1f GB" % (psutil.virtual_memory().total / 1e9))
    except ImportError:
        print("RAM: pasang psutil atau lihat spesifikasi perangkat")
    # random seed: salin nilai random_state yang Anda pakai di RandomForest/SVC/SMOTE/split

# ---------- 2. Uji McNemar eksak (binomial dua sisi) ----------
def mcnemar_exact(y, pred_a, pred_b):
    y, a, b = map(np.asarray, (y, pred_a, pred_b))
    n10 = int(np.sum((a == y) & (b != y)))   # A benar, B salah
    n01 = int(np.sum((a != y) & (b == y)))   # A salah, B benar
    p = 1.0 if n10 + n01 == 0 else binomtest(min(n10, n01), n10 + n01, 0.5).pvalue
    return n10, n01, p

# ---------- 3. Bootstrap berpasangan (selisih metrik A - B) ----------
def macro_f1(y, p):
    return f1_score(y, p, labels=np.unique(y), average="macro", zero_division=0)

def bootstrap_diff(y, pred_a, pred_b, metric=accuracy_score, B=10000, seed=42):
    y, a, b = map(np.asarray, (y, pred_a, pred_b))
    rng = np.random.default_rng(seed)
    n = len(y); d = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)
        d[i] = metric(y[idx], a[idx]) - metric(y[idx], b[idx])
    lo, hi = np.percentile(d, [2.5, 97.5])
    return metric(y, a) - metric(y, b), lo, hi

if __name__ == "__main__":
    info_lingkungan()

Python 3.13.18 | scikit-learn 1.6.1 | numpy 2.4.6 | scipy 1.16.3
imbalanced-learn 0.14.2
OS: Linux-6.6.159+-x86_64-with-glibc2.39 | CPU: x86_64 | core: 2
RAM: 13.6 GB
